In [ ]:
#Bloco 1 — Importar bibliotecas
import pandas as pd
import numpy as np

In [ ]:
#Bloco 2 — Carregar os dados
df = pd.read_excel('../data/Industrial_case_20251203.xlsx', 
                   sheet_name='1. OrdiniCliente')

print(df.shape)
print(df.head())
print(df.columns.tolist())

(52289, 7)
        Data Document_No Cliente    Item  Quantità  Classe MTS / MTO
0 2024-12-20   24-011070  I01257  530002    2022.0      53       MTS
1 2024-12-20   24-011071  I01229  500069    2023.0      50       MTS
2 2024-12-20   24-011071  I01229  500385    7065.0      50       MTS
3 2024-12-20   24-011071  I01229  500384    4057.0      50       MTS
4 2024-12-20   24-011072  M01629  530051    8035.0      53       MTS
['Data', 'Document_No', 'Cliente', 'Item', 'Quantità', 'Classe', 'MTS / MTO']


In [4]:
#Bloco 3 — Limpeza dos dados
# Remove devoluções (quantidade negativa ou zero)
df = df[df['Quantità'] > 0]

# Verifica valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())

# Verifica quantos itens únicos temos
print(f"\nItens únicos: {df['Item'].nunique()}")
print(f"Clientes únicos: {df['Cliente'].nunique()}")
print(f"Período: {df['Data'].min()} a {df['Data'].max()}")

# Verifica valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())

# Verifica quantos itens únicos temos
print(f"\nItens únicos: {df['Item'].nunique()}")
print(f"Clientes únicos: {df['Cliente'].nunique()}")
print(f"Período: {df['Data'].min()} a {df['Data'].max()}")

Valores nulos por coluna:
Data           0
Document_No    0
Cliente        0
Item           0
Quantità       0
Classe         0
MTS / MTO      0
dtype: int64

Itens únicos: 1388
Clientes únicos: 455
Período: 2023-01-02 00:00:00 a 2024-12-20 00:00:00
Valores nulos por coluna:
Data           0
Document_No    0
Cliente        0
Item           0
Quantità       0
Classe         0
MTS / MTO      0
dtype: int64

Itens únicos: 1388
Clientes únicos: 455
Período: 2023-01-02 00:00:00 a 2024-12-20 00:00:00


In [7]:
#Bloco 4 — Construir a série temporal mensal por item
# Cria coluna de ano-mês
df['YearMonth'] = df['Data'].dt.to_period('M')

# Agrega quantidade por item e mês
monthly = df.groupby(['Item', 'YearMonth'])['Quantità'].sum().reset_index()

# Cria o range completo de 24 meses
all_months = pd.period_range('2023-01', '2024-12', freq='M')

# Para cada item, garante que todos os 24 meses existem
# Meses sem pedido recebem zero
monthly = (
    monthly
    .set_index(['Item', 'YearMonth'])
    .reindex(
        pd.MultiIndex.from_product(
            [df['Item'].unique(), all_months],
            names=['Item', 'YearMonth']
        ),
        fill_value=0
    )
    .reset_index()
)

print(f"Shape da série temporal: {monthly.shape}")
print(monthly.head(10))

Shape da série temporal: (33312, 3)
     Item YearMonth  Quantità
0  530002   2023-01  173628.0
1  530002   2023-02  131314.0
2  530002   2023-03  173790.0
3  530002   2023-04  165738.0
4  530002   2023-05  120471.0
5  530002   2023-06  119348.0
6  530002   2023-07   91301.0
7  530002   2023-08   17116.0
8  530002   2023-09  153118.0
9  530002   2023-10  149716.0


In [17]:
#Bloco 5 — Calcular as features por item
features = []

for item, group in monthly.groupby('Item'):
    serie = group['Quantità'].values
    
    # Períodos com demanda maior que zero
    nonzero = serie[serie > 0]
    n_periods = len(serie)  # 24
    n_nonzero = len(nonzero)
    
    # Evita divisão por zero para itens sem nenhuma demanda
    if n_nonzero == 0:
        continue
    
    # Features de demanda
    avg_monthly    = serie.mean()
    std_monthly    = serie.std()
    cv             = std_monthly / avg_monthly if avg_monthly > 0 else 0
    cv2            = cv ** 2
    adi            = n_periods / n_nonzero
    zero_ratio     = 1 - (n_nonzero / n_periods)
    d_annual       = serie.sum() / 2  # 2 anos de dados
    avg_order_size = nonzero.mean()
    
    # Tendência (regressão linear simples)
    x = np.arange(n_periods)
    trend_slope = np.polyfit(x, serie, 1)[0]
    
    features.append({
        'Item':           item,
        'avg_monthly':    round(avg_monthly, 2),
        'std_monthly':    round(std_monthly, 2),
        'CV':             round(cv, 4),
        'CV2':            round(cv2, 4),
        'ADI':            round(adi, 4),
        'zero_ratio':     round(zero_ratio, 4),
        'D_annual':       round(d_annual, 2),
        'avg_order_size': round(avg_order_size, 2),
        'trend_slope':    round(trend_slope, 4),
    })

features_df = pd.DataFrame(features)
print(f"Items com features calculadas: {len(features_df)}")
print(features_df.head())

Items com features calculadas: 1388
     Item  avg_monthly  std_monthly      CV      CV2   ADI  zero_ratio  \
0  300007        50.71       243.19  4.7958  23.0000  24.0      0.9583   
1  300019       574.46      1793.44  3.1220   9.7466   8.0      0.8750   
2  300021         8.79        42.16  4.7958  23.0000  24.0      0.9583   
3  300027        28.58       137.08  4.7958  23.0000  24.0      0.9583   
4  300042       169.92       621.64  3.6585  13.3847   8.0      0.8750   

   D_annual  avg_order_size  trend_slope  
0     608.5         1217.00       6.8787  
1    6893.5         4595.67    -115.0743  
2     105.5          211.00      -2.1100  
3     343.0          686.00       5.6670  
4    2039.0         1359.33     -27.6165  


In [18]:
# Número de clientes distintos por item
n_clientes = (df.groupby('Item')['Cliente']
                .nunique()
                .reset_index()
                .rename(columns={'Cliente': 'n_clientes'}))

# Número de pedidos distintos por item
n_pedidos = (df.groupby('Item')['Document_No']
               .nunique()
               .reset_index()
               .rename(columns={'Document_No': 'n_pedidos'}))

# Classe do item (pega a mais frequente)
classe = (df.groupby('Item')['Classe']
            .agg(lambda x: x.mode()[0])
            .reset_index()
            .rename(columns={'Classe': 'Classe'}))

# Junta tudo no features_df
features_df = (features_df
               .merge(n_clientes, on='Item')
               .merge(n_pedidos,  on='Item')
               .merge(classe,     on='Item'))

print(f"Shape final: {features_df.shape}")
print(features_df.head(10))

Shape final: (1388, 13)
     Item  avg_monthly  std_monthly      CV      CV2      ADI  zero_ratio  \
0  300007        50.71       243.19  4.7958  23.0000  24.0000      0.9583   
1  300019       574.46      1793.44  3.1220   9.7466   8.0000      0.8750   
2  300021         8.79        42.16  4.7958  23.0000  24.0000      0.9583   
3  300027        28.58       137.08  4.7958  23.0000  24.0000      0.9583   
4  300042       169.92       621.64  3.6585  13.3847   8.0000      0.8750   
5  300082       177.58       851.66  4.7958  23.0000  24.0000      0.9583   
6  300091       783.17       658.31  0.8406   0.7066   1.2632      0.2083   
7  300092       258.54       620.03  2.3982   5.7513   3.4286      0.7083   
8  300111        99.00       414.97  4.1916  17.5694  12.0000      0.9167   
9  300120       490.25       713.32  1.4550   2.1171   2.4000      0.5833   

   D_annual  avg_order_size  trend_slope  n_clientes  n_pedidos  Classe  
0     608.5         1217.00       6.8787           1  

In [21]:
#Bloco 7 — Salvar a feature table
features_df.to_excel('../outputs/feature_table.xlsx', index=False)
print("Feature table salva com sucesso.")
features_df.to_csv('../outputs/feature_table.csv', index=False)
print("Feature table salva em CSV também.")

Feature table salva com sucesso.
Feature table salva em CSV também.


In [ ]:
features_df.to_csv('../outputs/feature_table.csv', index=False)
print("Feature table salva em CSV também.")